# tocandira: label every position again

Searches the positions of games already on Drive again, as a function of the position only:
for every position and depth, a fresh engine (one thread, 16 MiB table, root order seeded
by the position) searches it. The same position gets the same score in any game, order or
session. Output per shard: the relabeled games (`data/<OUT_M>/<source>-shard-NNN.jsonl.gz`)
and their radius 6 encoding (`data/<OUT_R6>/<source>-shard-NNN.npz`, layout version 3 with one
`score<d>` per hand eval depth, `score_net5` for the net and `opening`), so 02_train reads
it with `DATASET = "m"` and encodes nothing.

One notebook file per parallel run: save a copy in Drive per instance, set `INSTANCE` to 0
in one copy and 1 in the other. Instances take disjoint shards; a finished shard (its
`.jsonl.gz` exists) is skipped, so a disconnected session just gets re-run. Generation 1
(net labels only) takes about 12-15 h for 76 shards on two v6e-1, so about 20 min per n4j
shard per v6e-1.

CPU bound: pick the runtime with the most CPUs (v6e-1: 44, v5e-1: 24).

In [ ]:
INSTANCE, N_INSTANCES = 0, 2   # this copy does tasks with index % N_INSTANCES == INSTANCE
# data/<folder> with shard-NNN.jsonl.gz, the depth the games were played at, which shards;
# in order, most recent games first, so a stopped run leaves the most useful part.
# (n4j 0-19 were labeled locally at the same definition.)
SOURCES = [("n4", 4, range(60)), ("n4j", 4, range(20, 76)), ("d4", 4, range(30)), ("d5", 5, range(30))]
DEPTHS = ""  # hand eval depths; "" for net labels only (see docs/pipeline.md)
# Generation 2: the labeler l/lam09-s0 with deterministic search options at depth 5 (net5):
# as good as plain depth 5 at a third of the cost; depth 6 trained no better (2026-10-21).
NNUE_LABEL = "/content/drive/MyDrive/tocandira/models/l/lam09-s0/weights.bin"
NNUE_DEPTHS = "5"
NNUE_OPTIONS = "--null-move-pruning --lmr --killers --rfp=100"  # part of the label's definition, recorded in its labeler
SCALARS = True  # the strategy scalars in the same pass (no search, one replay per game)
DATA = "/content/drive/MyDrive/tocandira/data"
# Output folders under DATA; m and mr6 hold other labels of the same shards under the same
# file names, which would count as done.
OUT_M, OUT_R6 = "g2m", "g2mr6"

import os
print("cpus:", os.cpu_count())

In [ ]:
import time

from google.colab import drive

drive.mount("/content/drive")
# Instance 0 creates the folders and instance 1 waits for them: two sessions creating a
# folder at once leave two Drive folders of that name, each with half the shards.
if INSTANCE == 0:
    os.makedirs(f"{DATA}/{OUT_M}", exist_ok=True)
    os.makedirs(f"{DATA}/{OUT_R6}", exist_ok=True)
for folder in [f"{DATA}/{OUT_M}", f"{DATA}/{OUT_R6}"]:
    while not os.path.isdir(folder):
        print("waiting for instance 0 to create", folder)
        time.sleep(30)

In [ ]:
%%bash
set -e
if ! command -v cargo >/dev/null 2>&1; then curl -sSf https://sh.rustup.rs | sh -s -- -y -q >/dev/null; fi
source "$HOME/.cargo/env"
cd /content
# Cloned to /content/repo, not /content/tocandira: a folder named like the package would shadow it on sys.path.
if [ -d repo ]; then (cd repo && git pull -q && git submodule update --init -q); else git clone -q --recurse-submodules https://github.com/abstratovcm/tocandira repo; fi
cd /content/repo/engine/nokamute
CARGO_PROFILE_RELEASE_LTO=true CARGO_PROFILE_RELEASE_CODEGEN_UNITS=1 cargo build --release -q
# Not an editable install: its .pth file would only be read when the kernel restarts.
cd /content/repo && pip install -q .
./engine/nokamute/target/release/nokamute --help | head -1

In [ ]:
import gzip
import shutil
import subprocess
import time

WORKERS = os.cpu_count() or 2
os.environ["TOCANDIRA_ENGINE"] = "/content/repo/engine/nokamute/target/release/nokamute"
tasks = [(folder, depth, shard) for folder, depth, shards in SOURCES for shard in shards]
for folder, depth, shard in tasks[INSTANCE::N_INSTANCES]:
    name = f"{folder}-shard-{shard:03d}"
    npz, gz = f"{DATA}/{OUT_R6}/{name}.npz", f"{DATA}/{OUT_M}/{name}.jsonl.gz"
    if os.path.exists(gz):
        print("exists, skipping", gz)
        continue
    t0 = time.time()
    jsonl = f"/content/{name}.jsonl"
    subprocess.run(
        ["hive-label", "--relabel", f"{DATA}/{folder}/shard-{shard:03d}.jsonl.gz",
         "--depths", DEPTHS, "--source-depth", str(depth), "--workers", str(WORKERS),
         "--out", jsonl,
         *(["--nnue", NNUE_LABEL, "--nnue-depths", NNUE_DEPTHS, "--nnue-options", NNUE_OPTIONS]
           if NNUE_LABEL else []),
         *(["--scalars"] if SCALARS else [])],
        check=True,
    )
    subprocess.run(["hive-encode", jsonl, "--radius", "6", "--out", "/content/shard.npz",
                    "--workers", str(WORKERS)], check=True)
    # The npz first and the move lists last, each under a temporary name: 02_train only
    # sees finished shards (it globs the .jsonl.gz) and never encodes one itself.
    shutil.copyfile("/content/shard.npz", npz + ".part")
    os.replace(npz + ".part", npz)
    with open(jsonl, "rb") as src, gzip.open(gz + ".part", "wb") as dst:
        shutil.copyfileobj(src, dst)
    os.replace(gz + ".part", gz)
    os.remove(jsonl)
    os.remove("/content/shard.npz")
    print(f"{name}: {time.time() - t0:.0f}s -> {gz}")

Reference: 14 workers on a Ryzen 7 5700G label about 16 positions/s at depths 3+4+5 (depth
5 is nearly all of it), about 50/s on v6e-1: a depth 4 shard (77k positions) takes about 26
minutes there, a depth 5 shard (49k) about 16; v5e-1 about half that speed. Two runs of the
same shard give identical files.